### Check RNN fit NLL history
- To check if used n_epochs is sufficient

In [ ]:
import mab_subjects
import pandas as pd
import numpy as np

exps = (
    mab_subjects.unstruc.p8020_good_intact_sess
    + mab_subjects.struc.p8020_good_intact_sess
    + mab_subjects.unstruc.p8020_lesion_OFC_post_sess
    + mab_subjects.struc.p8020_lesion_OFC_post_sess
    + mab_subjects.unstruc.p8020_lesion_mPFC_post_sess
    + mab_subjects.struc.p8020_lesion_mPFC_post_sess
)

# exps = mab_subjects.struc.p8020_intact_Haaland + mab_subjects.struc.p8020_intact_Xavi

nll_df = []
for e, exp in enumerate(exps):
    print(exp.sub_name)
    task = exp.b2a
    task.auto_block_window_ids()

    if exp.lesion_tag == "intact":
        start_date = task.datetime[0]
        stop_date = task.datetime[-1]
        n_days = pd.Timedelta(stop_date - start_date).days
        if n_days > 30:
            task = task.filter_by_datetime(start=start_date + pd.Timedelta(days=30))

    rnn_fit1, rnn_fit2, rnn_fit3 = exp.rnn_fit1, exp.rnn_fit2, exp.rnn_fit3

    exp_dict = {
        "steps": np.arange(500) + 1,
        "nll_1": rnn_fit1.nll_history,
        "nll_2": rnn_fit2.nll_history,
        "nll_3": rnn_fit3.nll_history,
    }
    exp_dict.update(exp.common_kwargs)

    nll_df.append(pd.DataFrame(exp_dict))

nll_df = pd.concat(nll_df, axis=0, ignore_index=True)

mab_subjects.GroupData().save(nll_df, "nll_history_rnn_fit")

In [ ]:
import matplotlib.pyplot as plt
from statplotannot.plots import Fig, fix_legend, SeabornPlotter
from mab_colors import Palette2Arm
import mab_subjects

df = mab_subjects.GroupData().nll_history_rnn_fit.latest
df = df[df["lesion"].str.contains("intact")]

fig = Fig(5, 4, fontsize=12)

y_names = ["nll_1", "nll_2", "nll_3"]

for i, y in enumerate(y_names):
    ax = fig.subplot(fig.gs[i])

    SeabornPlotter(data=df, x="steps", y=y, hue="group", ax=ax).lineplot(
        palette=Palette2Arm().as_dict(), errorbar="se"
    )

    ax.grid(axis="y", zorder=-1, alpha=0.5)
    fix_legend(ax, only_labels=True)

# figpath = mab_subjects.figpath / "perf_animal_vs_rnn_fit.pdf"
# fig.savefig(figpath)

### Simulate RNN fit and compare performance

In [ ]:
import mab_subjects
import pandas as pd
import numpy as np

exps = (
    mab_subjects.unstruc.p8020_good_intact_sess
    + mab_subjects.struc.p8020_good_intact_sess
    + mab_subjects.unstruc.p8020_lesion_OFC_post_sess
    + mab_subjects.struc.p8020_lesion_OFC_post_sess
    + mab_subjects.unstruc.p8020_lesion_mPFC_post_sess
    + mab_subjects.struc.p8020_lesion_mPFC_post_sess
)

# exps = mab_subjects.struc.p8020_intact_Haaland + mab_subjects.struc.p8020_intact_Xavi

perf_df = []
for e, exp in enumerate(exps):
    print(exp.sub_name)
    task = exp.b2a
    task.auto_block_window_ids()

    if exp.lesion_tag == "intact":
        start_date = task.datetime[0]
        stop_date = task.datetime[-1]
        n_days = pd.Timedelta(stop_date - start_date).days
        if n_days > 30:
            task = task.filter_by_datetime(start=start_date + pd.Timedelta(days=30))

    task = task.filter_by_trials(min_trials=100, clip_max=100)
    task_perf = task.get_optimal_choice_probability()
    task_probs = task.probs[task.is_session_start, :]

    rnn_fit1, rnn_fit2, rnn_fit3 = exp.rnn_fit1, exp.rnn_fit2, exp.rnn_fit3

    perf_rnn_fit = []
    pseudo_r2 = []

    for rnn_fit in [rnn_fit1, rnn_fit2, rnn_fit3]:
        task_rnn_fit = rnn_fit.simulate(reward_schedule=task_probs, return_hidden=False)
        task_rnn_fit = task_rnn_fit.filter_by_trials(min_trials=100, clip_max=100)
        task_rnn_fit_perf = task_rnn_fit.get_optimal_choice_probability()
        nll = rnn_fit.nll_per_trial  # nats/trial
        null_nll = np.log(rnn_fit.n_ports)  # log(2) for 2-arm
        pseudo_r2.append(1 - nll / null_nll)  # 0 = chance, 1 = perfect
        perf_rnn_fit.append(task_rnn_fit_perf)

    exp_dict = {
        "trial_id": np.arange(100) + 1,
        "perf": task_perf,
        "perf_rnn_fit1": perf_rnn_fit[0],
        "perf_rnn_fit2": perf_rnn_fit[1],
        "perf_rnn_fit3": perf_rnn_fit[2],
        "pseudo_r2_1": pseudo_r2[0],
        "pseudo_r2_2": pseudo_r2[1],
        "pseudo_r2_3": pseudo_r2[2],
    }
    exp_dict.update(exp.common_kwargs)

    perf_df.append(pd.DataFrame(exp_dict))

perf_df = pd.concat(perf_df, axis=0, ignore_index=True)

mab_subjects.GroupData().save(perf_df, "perf_animal_vs_rnn_fit")

In [ ]:
import matplotlib.pyplot as plt
from statplotannot.plots import Fig, fix_legend, SeabornPlotter
from mab_colors import Palette2Arm
import mab_subjects

df = mab_subjects.GroupData().perf_animal_vs_rnn_fit.latest
df = df[df["lesion"].str.contains("intact")]

fig = Fig(5, 4, fontsize=12)

y_names = [
    "perf",
    "perf_rnn_fit1",
    "perf_rnn_fit2",
    "perf_rnn_fit3",
    "pseudo_r2_1",
    "pseudo_r2_2",
    "pseudo_r2_3",
]

for i, y in enumerate(y_names):
    ax = fig.subplot(fig.gs[i])

    if i < 4:
        SeabornPlotter(data=df, x="trial_id", y=y, hue="group", ax=ax).lineplot(
            palette=Palette2Arm().as_dict(), errorbar="se"
        )

        ax.set_ylim(0.4, 0.9)
        ax.grid(axis="y", zorder=-1, alpha=0.5)
    else:
        SeabornPlotter(data=df, x="group", y=y, hue="group", ax=ax).boxplot_filled(
            palette=Palette2Arm().as_dict()
        )
        ax.set_ylim(0.6, 1.0)
        ax.axhline(0.693, color="k", ls="--", lw=1, alpha=0.5)  # chance level for 2-arm
    fix_legend(ax, only_labels=True)

    ax.set_title("Animal" if y == "perf" else "RNN Fit")

figpath = mab_subjects.figpath / "perf_animal_vs_rnn_fit.pdf"
fig.savefig(figpath)

### PCA evolution of RNN fits

In [ ]:
import mab_subjects
import pandas as pd
import numpy as np
import itertools
from sklearn.decomposition import PCA

exps = (
    mab_subjects.unstruc.p8020_good_intact_sess
    + mab_subjects.struc.p8020_good_intact_sess
    + mab_subjects.unstruc.p8020_lesion_OFC_post_sess
    + mab_subjects.struc.p8020_lesion_OFC_post_sess
    + mab_subjects.unstruc.p8020_lesion_mPFC_post_sess
    + mab_subjects.struc.p8020_lesion_mPFC_post_sess
)

probs = np.array([0.2, 0.3, 0.4, 0.6, 0.7, 0.8])
prob_pairs = np.array(list(itertools.permutations(probs, 2)))
prob_diffs = np.diff(prob_pairs, axis=1).flatten()
pca = PCA(n_components=7)

pca_df = []
for e, exp in enumerate(exps):
    print(exp.sub_name)
    task = exp.b2a
    task.auto_block_window_ids()

    if exp.lesion_tag == "intact":
        start_date = task.datetime[0]
        stop_date = task.datetime[-1]
        n_days = pd.Timedelta(stop_date - start_date).days
        if n_days > 30:
            task = task.filter_by_datetime(start=start_date + pd.Timedelta(days=30))

    task = task.filter_by_trials(min_trials=100, clip_max=100)

    rnn_fit = exp.rnn_fit3
    hidden_states = rnn_fit.simulate(
        reward_schedule=prob_pairs,
        return_hidden=True,
        min_trials_per_block=100,
        max_trials_per_block=100,
        prob_switch=100,
        n_block_min=1,
        n_block_max=1,
    )[1]
    hidden_pca = pca.fit_transform(hidden_states)
    final_state = hidden_pca[99::100, :]

    exp_dict = {
        "pca0": final_state[:, 0],
        "pca1": final_state[:, 1],
        "pca2": final_state[:, 2],
        "prob_diff": prob_diffs,
    }
    exp_dict.update(exp.common_kwargs)

    pca_df.append(pd.DataFrame(exp_dict))

pca_df = pd.concat(pca_df, axis=0, ignore_index=True)
mab_subjects.GroupData().save(pca_df, "pca_rnn_fit")

In [ ]:
import matplotlib.pyplot as plt

_, ax = plt.subplots(1, 1)

ax.plot(hidden_pca[:, 0])
ax.plot(hidden_pca[:, 1])
# ax.plot(hidden_pca[:, 4])

In [ ]:
import matplotlib.pyplot as plt

_, ax = plt.subplots(1, 1)

ax.scatter(hidden_pca[:, 0], hidden_pca[:, 1], s=3)
ax.set_ylabel("PCA 1")
ax.set_xlabel("PCA 0")
# ax.plot(hidden_pca[:, 1])

In [ ]:
import mab_subjects
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statplotannot.plots import Fig

fig = Fig(4, 3, fontsize=12)

df = mab_subjects.GroupData().pca_rnn_fit.latest
df = df[df["lesion"].str.contains("intact")]
df["difficulty"] = pd.cut(
    df["prob_diff"],
    bins=[-0.7, -0.3, 0, 0.3, 0.7],
    labels=["left", "left-biased", "right-biased", "right"],
)
# df = df[df["difficulty"].isin(["left", "right"])]


for p, paradigm in enumerate(["8020"]):
    df_paradigm = df[df["paradigm"] == paradigm]
    for g, group in enumerate(["unstruc", "struc"]):
        df_group = df_paradigm[df_paradigm["group"] == group]
        ax = fig.subplot(fig.gs[p, g])
        # ax.scatter(
        #     df_group["pca0"],
        #     df_group["pca1"],
        #     df_group["pca2"],
        #     c=df_group["difficulty"].cat.codes,
        #     cmap="coolwarm",
        #     alpha=0.7,
        # )
        sns.scatterplot(
            data=df_group,
            x="pca0",
            y="pca1",
            hue="difficulty",
            palette=["blue", "gray", "gray", "red"],  # "Set1",
            alpha=0.5,
            size=3,
            ax=ax,
        )

        ax.legend_.remove()

        ax.set_title(f"{paradigm} {group}")

### Mean PCA

In [ ]:
import mab_subjects
import pandas as pd
import numpy as np
import itertools
from sklearn.decomposition import PCA

exps = (
    mab_subjects.unstruc.p8020_good_intact_sess
    + mab_subjects.struc.p8020_good_intact_sess
    # + mab_subjects.unstruc.p8020_lesion_OFC_post_sess
    # + mab_subjects.struc.p8020_lesion_OFC_post_sess
    # + mab_subjects.unstruc.p8020_lesion_mPFC_post_sess
    # + mab_subjects.struc.p8020_lesion_mPFC_post_sess
)

probs = np.array([0.2, 0.3, 0.4, 0.6, 0.7, 0.8])
prob_pairs = np.array(list(itertools.permutations(probs, 2)))
prob_diffs = np.diff(prob_pairs, axis=1).flatten()
pca = PCA(n_components=7)

pca_df = []
for e, exp in enumerate(exps):
    print(exp.sub_name)
    task = exp.b2a
    task.auto_block_window_ids()

    if exp.lesion_tag == "intact":
        start_date = task.datetime[0]
        stop_date = task.datetime[-1]
        n_days = pd.Timedelta(stop_date - start_date).days
        if n_days > 30:
            task = task.filter_by_datetime(start=start_date + pd.Timedelta(days=30))

    task = task.filter_by_trials(min_trials=100, clip_max=100)

    rnn_fit = exp.rnn_fit3
    hidden_states = rnn_fit.simulate(
        reward_schedule=prob_pairs,
        return_hidden=True,
        min_trials_per_block=100,
        max_trials_per_block=100,
        prob_switch=100,
        n_block_min=1,
        n_block_max=1,
    )[1]
    hidden_pca = pca.fit_transform(hidden_states)
    mean_pca = hidden_pca.reshape(30, 100, 7).transpose(1, 2, 0).mean(axis=-1)
    final_state = hidden_pca[99::100, :]

    exp_dict = {
        "trial_id": np.arange(100) + 1,
        "mean_pca0": mean_pca[:, 0],
        "mean_pca1": mean_pca[:, 1],
        "mean_pca2": mean_pca[:, 2],
    }
    exp_dict.update(exp.common_kwargs)

    pca_df.append(pd.DataFrame(exp_dict))

pca_df = pd.concat(pca_df, axis=0, ignore_index=True)
mab_subjects.GroupData().save(pca_df, "pca_mean_rnn_fit")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from statplotannot.plots import Fig
from mab_colors import Palette2Arm

fig = Fig(5, 3, fontsize=12)

df = mab_subjects.GroupData().pca_mean_rnn_fit.latest

for y, pca in enumerate(["mean_pca0", "mean_pca1", "mean_pca2"]):
    ax = fig.subplot(fig.gs[y])
    sns.lineplot(
        data=df,
        x="trial_id",
        y=pca,
        hue="group",
        palette=Palette2Arm().as_dict(),  # "Set1",
        alpha=0.5,
        size=3,
        ax=ax,
        errorbar="se",
    )

    ax.legend_.remove()

    # ax.set_title(f"{paradigm} {group}")